# 💅 ระบบแนะนำสีทาเล็บจากโทนผิว — เวอร์ชัน Colab (GUI)
**CP352301 Script Programming — Final Project**

วิเคราะห์โทนผิวรอบเล็บจากรูป (OpenCV / CIE Lab / ITA) แล้วแนะนำพาเลตสีทาเล็บ

### วิธีใช้
1. กดรันเซลล์ตามลำดับ (Shift+Enter) จากบนลงล่าง
2. เซลล์สุดท้ายจะมี **dropdown เลือกทรง/ความยาว** และปุ่มอัปโหลดรูป
3. กดปุ่ม → เลือกรูปมือ/เล็บ → ระบบจะแสดงโทนผิว + พาเลตสีที่แนะนำ

> โน้ตบุ๊กนี้เป็นแบบ **self-contained** (โค้ด OOP อยู่ในตัว) เปิดแล้วรันได้ทันที
> ส่วนเวอร์ชันรันบนเครื่อง (CLI) และเทสต์ อยู่ในโปรเจกต์ GitHub เดียวกัน

In [ ]:
# ติดตั้งไลบรารีสำหรับ Colab
!pip -q install ipywidgets pillow-heif pillow

In [ ]:
# (ไม่บังคับ) ดาวน์โหลดฟอนต์ไทยสำหรับใส่ข้อความบนภาพสรุป
import matplotlib, matplotlib.font_manager as fm
!wget -q -O /usr/local/share/fonts/thsarabun.ttf https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
FONT_PATH = '/usr/local/share/fonts/thsarabun.ttf'
try:
    fm.fontManager.addfont(FONT_PATH)
    matplotlib.rcParams['font.family'] = fm.FontProperties(fname=FONT_PATH).get_name()
    print('โหลดฟอนต์ไทยสำเร็จ')
except Exception as e:
    FONT_PATH = None
    print('ใช้ฟอนต์เริ่มต้น:', e)

In [ ]:
# ===== โค้ด OOP (Business Logic + Data Access) แบบ self-contained =====
import math, cv2 as cv, numpy as np

# ----- กฎพาเลต (data-driven: แก้/เพิ่มสีได้ที่นี่) -----
PALETTE_RULES = {
  "undertone": {
    "cool": [
      {"palette":["#C3C7FF","#EDEBFF"],"tags":["pastel","cool","minimal"],"note":"พาสเทลเย็น/มุกเงิน"},
      {"palette":["#FFB3D1","#9AD0F5"],"tags":["duo","cool"],"note":"ชมพูเย็น + ฟ้าอ่อน"},
      {"palette":["#FFFFFF","#BFC6D1"],"tags":["chrome","silver"],"note":"เงิน/โครมเงิน"},
      {"palette":["#E6E6FA","#FFB6C1"],"tags":["pastel","cool"],"note":"พาสเทลม่วงลาเวนเดอร์ + ชมพูเย็น"},
      {"palette":["#98FF98","#40E0D0"],"tags":["duo","cool"],"note":"มิ้นต์เขียว + ฟ้าคราม"},
      {"palette":["#8B0000","#0F52BA"],"tags":["jewel","cool"],"note":"แดงอมฟ้า + น้ำเงินอัญมณี"},
      {"palette":["#C0C0C0","#FFFFFF"],"tags":["metallic","cool"],"note":"เงินไอซี่ + ขาว"},
      {"palette":["#4B0082","#800080"],"tags":["plum","cool"],"note":"ม่วงพลัมเข้ม + ม่วงบริสุทธิ์"}
    ],
    "warm": [
      {"palette":["#F2C6A0","#E8A87C"],"tags":["nude","warm"],"note":"นู้ดพีช/คาราเมล"},
      {"palette":["#B8860B","#FFD700"],"tags":["gold","accent"],"note":"ทอง/กลิทเตอร์ทอง"},
      {"palette":["#6B8E23","#4B5320"],"tags":["olive","earthy"],"note":"เขียวมะกอก"},
      {"palette":["#B22222","#FFA07A"],"tags":["warm","classic"],"note":"แดงอิฐ + พีชอมส้ม"},
      {"palette":["#FFDB58","#FFD700"],"tags":["gold","warm"],"note":"เหลืองมัสตาร์ด + ทองอุ่น"},
      {"palette":["#CC5500","#B8860B"],"tags":["earthy","warm"],"note":"ส้มเข้ม + น้ำตาลทอง"},
      {"palette":["#FF7F50","#FFDAB9"],"tags":["coral","warm"],"note":"ปะการัง + พีชนู้ด"},
      {"palette":["#6B8E23","#808000"],"tags":["olive","warm"],"note":"เขียวมะกอก + เขียวอ่อนเอิร์ธโทน"}
    ],
    "neutral": [
      {"palette":["#F5E6DC","#E3C4B5"],"tags":["nude","neutral"],"note":"นู้ดชมพู/โทนกลาง"},
      {"palette":["#C00021","#8B0016"],"tags":["classic","red"],"note":"แดงคลาสสิก"},
      {"palette":["#FDF5F8","#E2D7FF"],"tags":["soft","pastel"],"note":"พาสเทลนุ่ม"},
      {"palette":["#FF0000","#800020"],"tags":["classic","neutral"],"note":"แดงคลาสสิก + แดงไวน์"},
      {"palette":["#000080","#50C878"],"tags":["bold","neutral"],"note":"น้ำเงินเนวี่ + เขียวมรกต"},
      {"palette":["#483C32","#C0C0C0"],"tags":["nude","metallic"],"note":"นู้ดเทา + เงินเมทัลลิก"},
      {"palette":["#F5CBA7","#FADBD8"],"tags":["blush","neutral"],"note":"ชมพูบลัช + ชมพูอ่อน"},
      {"palette":["#36454F","#808080"],"tags":["gray","neutral"],"note":"เทาชาร์โคล + เทากลาง"}
    ]
  },
  "shape_length": {
    "micro_french":{"palette":["#F2EFEA"],"tags":["micro-french"],"note":"micro-French ขอบบางเพราะเล็บสั้น"},
    "oval_almond":{"palette":["#FBE9E7","#FFE0B2"],"tags":["ombre","vertical"],"note":"ombré แนวตั้งสีละมุนนุ่ม"},
    "square":{"palette":["#FFFFFF","#000000"],"tags":["geo","contrast"],"note":"เส้นหนา/กราฟิกขาวดำ"},
    "long":{"palette":["#2E2A2A","#D4AF37"],"tags":["statement","3d"],"note":"ปั้นนูน 3D สำหรับเล็บยาว"}
  }
}

class Suggestion:
    def __init__(self, palette, tags, note):
        self.palette, self.tags, self.note = list(palette), list(tags), note
    def key(self): return tuple(self.palette) + tuple(self.tags)

class SkinAnalyzer:
    LOWER = np.array([0,133,77], np.uint8); UPPER = np.array([255,173,127], np.uint8)
    def skin_mask(self, bgr):
        m = cv.inRange(cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb), self.LOWER, self.UPPER)
        k = cv.getStructuringElement(cv.MORPH_ELLIPSE,(5,5))
        m = cv.morphologyEx(m, cv.MORPH_OPEN, k, iterations=2)
        m = cv.morphologyEx(m, cv.MORPH_CLOSE, k, iterations=2)
        num, labels, stats, _ = cv.connectedComponentsWithStats(m, connectivity=8)
        if num > 1:
            largest = 1 + int(np.argmax(stats[1:, cv.CC_STAT_AREA]))
            m = np.where(labels==largest, 255, 0).astype(np.uint8)
        return m
    def estimate_undertone(self, bgr, mask):
        sel = mask > 0
        if int(sel.sum()) < 500: return "unknown", 0.0
        lab = cv.cvtColor(bgr, cv.COLOR_BGR2Lab)
        L = lab[...,0][sel].mean()*(100/255.0); b = lab[...,2][sel].mean()-128
        ita = math.degrees(math.atan2((L-50), b if abs(b)>1e-3 else 1e-3))
        tone = "cool" if ita>=28 else ("warm" if ita<=10 else "neutral")
        return tone, ita

class NailRecommender:
    def __init__(self, rules, max_results=6): self.rules, self.max_results = rules, max_results
    def _b(self, lst): return [Suggestion(d["palette"], d["tags"], d["note"]) for d in lst]
    def suggest(self, undertone, shape, length):
        recs = self._b(self.rules["undertone"].get(undertone, self.rules["undertone"]["neutral"]))
        sl = self.rules["shape_length"]
        if shape in ["round","short"] or length=="short": recs = self._b([sl["micro_french"]]) + recs
        elif shape in ["oval","almond"]: recs = self._b([sl["oval_almond"]]) + recs
        elif shape in ["square","squoval"]: recs = self._b([sl["square"]]) + recs
        else: recs = self._b([sl["long"]]) + recs
        seen, uniq = set(), []
        for s in recs:
            if s.key() not in seen: seen.add(s.key()); uniq.append(s)
        return uniq[:self.max_results]

def hex_to_bgr(hx):
    hx = hx.lstrip('#'); return (int(hx[4:6],16), int(hx[2:4],16), int(hx[0:2],16))

def show_palette_row(title, palette):
    import matplotlib.pyplot as plt
    W,H = 400,50; row = np.ones((H,W,3),np.uint8)*255; n=len(palette); sw=W//n
    for i,hx in enumerate(palette): row[:, i*sw:(i+1)*sw] = hex_to_bgr(hx)
    plt.figure(figsize=(W/120,H/120)); plt.imshow(cv.cvtColor(row,cv.COLOR_BGR2RGB))
    plt.axis('off'); plt.title(title); plt.show()

print("โหลดคลาสเรียบร้อย: SkinAnalyzer, NailRecommender, Suggestion")

In [ ]:
# ===== GUI: เลือกทรง/ความยาว + อัปโหลดรูป แล้วแนะนำสี =====
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

analyzer = SkinAnalyzer()
recommender = NailRecommender(PALETTE_RULES)

shape_dd  = widgets.Dropdown(options=["round","oval","almond","square","squoval","stiletto"], value="oval", description="ทรงเล็บ:")
length_dd = widgets.Dropdown(options=["short","medium","long"], value="medium", description="ความยาว:")
btn = widgets.Button(description="อัปโหลดรูปเล็บและแนะนำสี", button_style='primary')
out = widgets.Output()
display(shape_dd, length_dd, btn, out)

def on_click(_):
    with out:
        clear_output()
        uploaded = files.upload()
        if not uploaded:
            print("ไม่พบไฟล์"); return
        fname = next(iter(uploaded))
        img = cv.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv.IMREAD_COLOR)
        if img is None:
            print("เปิดไฟล์รูปไม่ได้"); return
        if img.shape[1] > 900:
            s = 900/img.shape[1]; img = cv.resize(img,(900,int(img.shape[0]*s)))

        mask = analyzer.skin_mask(img)
        tone, ita = analyzer.estimate_undertone(img, mask)
        recs = recommender.suggest(tone, shape_dd.value, length_dd.value)

        fig, axs = plt.subplots(1,2, figsize=(11,4))
        axs[0].imshow(cv.cvtColor(img,cv.COLOR_BGR2RGB)); axs[0].set_title("ภาพต้นฉบับ"); axs[0].axis('off')
        axs[1].imshow(mask, cmap='gray'); axs[1].set_title("ประมาณผิว (mask)"); axs[1].axis('off')
        plt.show()

        print(f"โทนผิว (Undertone): {tone}  (ITA ≈ {ita:.1f})")
        print(f"ทรงเล็บ: {shape_dd.value} | ความยาว: {length_dd.value}")
        print(f"\n🎨 คำแนะนำพาเลต/สไตล์ (สูงสุด {recommender.max_results}):")
        for i, s in enumerate(recs, 1):
            print(f"{i}. {s.note}  | tags={s.tags} | palette={s.palette}")
            show_palette_row(f"{i}) {s.note}", s.palette)

btn.on_click(on_click)